# 4. Train every parameter of Whisper-small

An epoch is one pass over every training recording. We train for three epochs and select the best model using validation loss. Test recordings never select the checkpoint.

## Imports and required earlier steps

Choose the **Whisper (project .venv)** kernel. This cell imports the libraries and loads missing settings/prerequisites. Training stays disabled until you explicitly enable it.

In [ ]:
from pathlib import Path
import json
import torch

# Load settings when this notebook starts in a fresh kernel.
if "DATA_DIR" not in globals():
    root = Path.cwd().resolve()
    while not (root / "pyproject.toml").is_file():
        if root.parent == root:
            raise FileNotFoundError("Start the kernel inside the AI & ML project.")
        root = root.parent
    setup = json.loads((root / "notebooks/whisper/setup.ipynb").read_text())
    for cell in setup["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())

if "train_loader" not in globals():
    run_notebook("03_batching.ipynb")

## 1. Measure validation loss

Validation predicts reference transcripts without updating weights. Padding is ignored, and every real target token contributes equally.

In [ ]:
def validation_loss():
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch in validation_loader:
            token_count = int(batch["labels"].ne(-100).sum())
            batch = {name: tensor.to(DEVICE) for name, tensor in batch.items()}
            loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise ValueError("Validation loss is invalid.")
            total_loss += loss.item() * token_count
            total_tokens += token_count
    model.train()
    return total_loss / total_tokens

## 2. Train one epoch

`backward()` calculates gradients: how weights should change to lower loss. AdamW applies the update after 8 batches. The final partial window uses its actual number of tokens. Gradient clipping limits the size of an update.

In [ ]:
def train_one_epoch(epoch, optimizer, updates):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    total_tokens = 0
    window_tokens = 0

    for step, batch in enumerate(train_loader, start=1):
        # Move the batch to the GPU and predict its correct transcript.
        token_count = int(batch["labels"].ne(-100).sum())
        batch = {name: tensor.to(DEVICE) for name, tensor in batch.items()}
        loss = model(**batch).loss
        if not torch.isfinite(loss):
            raise ValueError("Training loss is invalid.")

        # Calculate gradients and add them to the current accumulation window.
        (loss * token_count).backward()
        total_loss += loss.item() * token_count
        total_tokens += token_count
        window_tokens += token_count

        # Update after 8 batches, or after the final partial group of batches.
        if step % ACCUMULATION_STEPS == 0 or step == len(train_loader):
            for parameter in model.parameters():
                if parameter.grad is not None:
                    parameter.grad.div_(window_tokens)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            window_tokens = 0
            updates += 1

            # Save a small progress report that can be read while training runs.
            progress = {"status": "training", "epoch": epoch + 1, "epochs": EPOCHS,
                        "batch": step, "batches": len(train_loader), "updates": updates,
                        "train_loss": total_loss / total_tokens}
            (RUN_DIR / "progress.json").write_text(json.dumps(progress, indent=2))
            with (RUN_DIR / "loss_history.csv").open("a") as log:
                log.write(f"{epoch + 1},{step},{updates},{total_loss / total_tokens}\n")
            print(f"Epoch {epoch + 1}/{EPOCHS} | batch {step}/{len(train_loader)} | "
                  f"updates {updates} | loss {total_loss / total_tokens:.4f}", flush=True)
    return total_loss / total_tokens, updates

## 3. Prepare the run folder and optimizer

Save the settings and exact split membership. A nonempty run folder is refused to protect previous experiments. AdamW is the algorithm that updates the weights.

In [ ]:
if RUN_TRAINING:
    if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
        raise FileExistsError("Choose a new RUN_NAME; this run folder already has files.")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    (RUN_DIR / "loss_history.csv").write_text("epoch,batch,updates,train_loss\n")

    settings = {
        "model_id": MODEL_ID, "language": LANGUAGE, "seed": SEED,
        "dataset_dir": str(DATA_DIR), "device": DEVICE,
        "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "accumulation_steps": ACCUMULATION_STEPS,
        "validation_fraction": VALIDATION_FRACTION, "max_new_tokens": MAX_NEW_TOKENS,
        "train_rows": len(train_df), "validation_rows": len(validation_df),
        "test_rows": len(test_df), "trainable_parameters": parameter_count,
    }
    (RUN_DIR / "settings.json").write_text(json.dumps(settings, indent=2))
    for name, table in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
        table[["audio_path", "text", "duration"]].to_csv(RUN_DIR / f"{name}_split.csv", index=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    history = []
    best_loss = float("inf")
    updates = 0
else:
    print("Training is disabled for this workflow check.")

## 4. Run three epochs and keep the best model

Each epoch trains on all training rows, measures validation loss, and updates the loss history. A better validation loss replaces `best/`.

In [ ]:
if RUN_TRAINING:
    for epoch in range(EPOCHS):
        train_loss, updates = train_one_epoch(epoch, optimizer, updates)
        progress = {"status": "validation", "epoch": epoch + 1, "epochs": EPOCHS,
                    "batch": len(train_loader), "batches": len(train_loader), "updates": updates,
                    "train_loss": train_loss}
        (RUN_DIR / "progress.json").write_text(json.dumps(progress, indent=2))
        current_validation_loss = validation_loss()

        history.append({"epoch": epoch + 1, "train_loss": train_loss,
                        "validation_loss": current_validation_loss, "updates": updates})
        (RUN_DIR / "history.json").write_text(json.dumps(history, indent=2))

        # Save model and processor only when validation improves.
        if current_validation_loss < best_loss:
            best_loss = current_validation_loss
            model.save_pretrained(RUN_DIR / "best")
            processor.save_pretrained(RUN_DIR / "best")
        print("Validation loss:", current_validation_loss, "| best:", best_loss, flush=True)
        if DEVICE == "mps":
            torch.mps.empty_cache()

## 5. Mark training complete

Progress is updated after each optimizer step. This final report is written only after all epochs finish.

In [ ]:
if RUN_TRAINING:
    progress = {"status": "finished", "epochs": EPOCHS, "updates": updates,
                "best_validation_loss": best_loss}
    (RUN_DIR / "progress.json").write_text(json.dumps(progress, indent=2))
    print("Training finished. Best checkpoint:", RUN_DIR / "best", flush=True)